In [1]:
!pip install fastapi uvicorn pyngrok nest-asyncio

In [4]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_TOKEN")
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")
GIST_ID = user_secrets.get_secret("GIST_ID_ENDPOINT_MARINE")

In [5]:
from datetime import datetime
import json
import requests

HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
}


def write_endpoint(
    endpoint: str,
    working: bool = True,
    last_checked: datetime | None = None,
) -> bool:
    if last_checked is None:
        last_checked = datetime.now()

    content = json.dumps(
        {
            "endpoint": endpoint,
            "working": working,
            "last_checked": str(last_checked),
        },
        indent=4,
    )

    payload = {"files": {"Endpoint.json": {"content": content}}}

    response = requests.patch(
        f"https://api.github.com/gists/{GIST_ID}",
        json=payload,
        headers=HEADERS,
    )

    return response.status_code == 200


def read_endpoint() -> dict:
    response = requests.get(
        f"https://api.github.com/gists/{GIST_ID}",
        headers=HEADERS,
    )
    response.raise_for_status()

    content = response.json()["files"]["Endpoint.json"]["content"]
    return json.loads(content)

In [8]:
read_endpoint()

{'endpoint': 'not working',
 'working': False,
 'last_checked': '2026-06-04 15:16:21.156356'}

In [9]:
import nest_asyncio
import uvicorn
from threading import Thread

from fastapi import FastAPI
from pyngrok import ngrok

from datetime import datetime

# Allow nested event loops in notebooks
nest_asyncio.apply()

app = FastAPI()


@app.get("/")
def root():
    return {"message": "Hello from Kaggle!"}


@app.get("/ping")
def ping():
    return {"Time": datetime.now(), "Status": "alive", "Static": "static"}


# Configure ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)


# Start FastAPI in a background thread
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)


server = Thread(target=run)
server.start()

# Create ngrok tunnel
public_url = ngrok.connect(8000)


print(f"Public URL: {public_url.public_url}")

# Update Endpoint for visibility
write_endpoint(public_url.public_url, True, datetime.now())

INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: https://6643-35-244-99-0.ngrok-free.app


True

INFO:     2405:201:4018:afdf:46ac:8a4a:3648:c96d:0 - "GET / HTTP/1.1" 200 OK
INFO:     2405:201:4018:afdf:46ac:8a4a:3648:c96d:0 - "GET /ping HTTP/1.1" 200 OK
